## **Persistance Example**

In [1]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

from typing import TypedDict
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [4]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [5]:
## Defining the State
class JokeState(TypedDict):
    ## Initialising the variables
    topic: str
    joke: str
    explanation: str

In [7]:
## Generating the joke using prompt
def generate_joke(state: JokeState) -> JokeState:
    ## Genrate a prompt 
    prompt = f"Generate a joke on the topic {state["topic"]}"
    response = llm.invoke(prompt).content

    return {"joke": response}

In [8]:
## Explanation of the joke
def generate_explanation(state: JokeState) -> JokeState:
    ## Genrating a prompt to explain
    prompt = f"Write an explanation for the joke - {state["joke"]}"
    response = llm.invoke(prompt).content

    return {"explanation": response}

In [9]:
## Buiding the graph
graph = StateGraph(JokeState)


## Add Nodes
graph.add_node("generate_joke", generate_joke)
graph.add_node("generate_explanation", generate_explanation)


## Add edges
graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke", "generate_explanation")
graph.add_edge("generate_explanation", END)


## Checkpointer definition
checkpointer = InMemorySaver()


## Compile the graph
workflow = graph.compile(checkpointer=checkpointer)

In [12]:
## Creating configuration
config = {"configurable": {"thread_id": "1"}}
result = workflow.invoke({"topic": "pizza"}, config=config)

In [13]:
print(result)

{'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'Why did the hipster burn the roof of his mouth?\n\nBecause he ate the pizza before it was cool.', 'extras': {'signature': 'EpYBCpMBAVjVB3tgbik3XpnSHfoD0ehNo7TaQQK6pBiaiW+dcQcfCz7yntNyYOte08wpIlXTVkFMvLVFeZAh9kf42skNl7QaitwGuHYUUx4r751Q5k6v1ioqrPOGVIGBD9i7AzlQsDqM6OQ2naYj4Y3trm4eVKjqFFnLLK0sUyksTnzWkzChYvvVL8gYEW78qisnfyhHB2or'}}], 'explanation': [{'type': 'text', 'text': 'This joke is a pun that plays on a double meaning of the word **"cool."**\n\n1. **Literal meaning (temperature):** Pizza is typically eaten hot, but if you eat it straight out of the oven before it has cooled down, you will burn the roof of your mouth. \n2. **Slang meaning (trendy/fashionable):** The word "hipster" refers to a subculture of people who pride themselves on being ahead of cultural trends. A running joke about hipsters is that they will only like music, fashion, or things *before* they become mainstream or popular—in other words, "before it was cool."\

In [14]:
result = workflow.invoke({"topic": "pizza"}, config=config)

joke = result["joke"][0]["text"]
explanation = result["explanation"][0]["text"]

print("Topic:", result["topic"])

print("\nJoke:")
print(joke)

print("\nExplanation:")
print(explanation)

Topic: pizza

Joke:
Why did the hipster burn the roof of his mouth?

He ate the pizza before it was cool.

Explanation:
This joke is a play on words (a double entendre) based on the slang meaning of the word "cool." 

Here is the breakdown:
1. **Literal meaning:** When someone eats hot food like pizza too quickly, they can burn the roof of their mouth because the food has not had time to cool down in temperature. 
2. **Slang meaning:** "Hipsters" are a cultural subculture stereotyped by their desire to be trendy, unique, and ahead of the curve—often liking bands, movies, or trends *before* they become mainstream or "cool."

**The punchline:** The hipster burned his mouth because he literally ate the pizza **before it was cool** (temperature-wise), which ironically aligns with his personality trait of liking everything before it is popular/cool.


In [15]:
workflow.get_state(config)

StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'Why did the hipster burn the roof of his mouth?\n\nHe ate the pizza before it was cool.', 'extras': {'signature': 'EpYBCpMBAVjVB3uKodlSxKn4KuKffIuPZvFbEA2KYZG01aDvwG329+pDUD9Ekaln4i21K0aEUL+Vclh7IsgiEfwrw8ZNu7WJYltvVomKlDaGbuMmBMj+vYswCJgALIUR5s+Db+dFcL1weltnCNeOOj49MaM3n8Fls7JMXd/Hegu5TNBtvvf9VoXDyn6mruzp6tUgv2AV5TCf'}}], 'explanation': [{'type': 'text', 'text': 'This joke is a play on words (a double entendre) based on the slang meaning of the word "cool." \n\nHere is the breakdown:\n1. **Literal meaning:** When someone eats hot food like pizza too quickly, they can burn the roof of their mouth because the food has not had time to cool down in temperature. \n2. **Slang meaning:** "Hipsters" are a cultural subculture stereotyped by their desire to be trendy, unique, and ahead of the curve—often liking bands, movies, or trends *before* they become mainstream or "cool."\n\n**The punchline:** The hipster burned hi

In [16]:
list(workflow.get_state_history(config))

[StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'Why did the hipster burn the roof of his mouth?\n\nHe ate the pizza before it was cool.', 'extras': {'signature': 'EpYBCpMBAVjVB3uKodlSxKn4KuKffIuPZvFbEA2KYZG01aDvwG329+pDUD9Ekaln4i21K0aEUL+Vclh7IsgiEfwrw8ZNu7WJYltvVomKlDaGbuMmBMj+vYswCJgALIUR5s+Db+dFcL1weltnCNeOOj49MaM3n8Fls7JMXd/Hegu5TNBtvvf9VoXDyn6mruzp6tUgv2AV5TCf'}}], 'explanation': [{'type': 'text', 'text': 'This joke is a play on words (a double entendre) based on the slang meaning of the word "cool." \n\nHere is the breakdown:\n1. **Literal meaning:** When someone eats hot food like pizza too quickly, they can burn the roof of their mouth because the food has not had time to cool down in temperature. \n2. **Slang meaning:** "Hipsters" are a cultural subculture stereotyped by their desire to be trendy, unique, and ahead of the curve—often liking bands, movies, or trends *before* they become mainstream or "cool."\n\n**The punchline:** The hipster burned h

In [18]:
for snapshot in workflow.get_state_history(config):

    state = snapshot.values

    print("=" * 50)
    print("Thread ID:", snapshot.config["configurable"]["thread_id"])
    print("Step:", snapshot.metadata["step"])
    print("Next Node:", snapshot.next)

    print("\nTopic:", state.get("topic", "N/A"))

    joke = state.get("joke")
    if joke:
        print("\nJoke:")
        print(joke[0]["text"])

    explanation = state.get("explanation")
    if explanation:
        print("\nExplanation:")
        print(explanation[0]["text"])

    print("=" * 50)

Thread ID: 1
Step: 9
Next Node: ()

Topic: pizza

Joke:
Why did the hipster burn the roof of his mouth?

He ate the pizza before it was cool.

Explanation:
This joke is a play on words (a double entendre) based on the slang meaning of the word "cool." 

Here is the breakdown:
1. **Literal meaning:** When someone eats hot food like pizza too quickly, they can burn the roof of their mouth because the food has not had time to cool down in temperature. 
2. **Slang meaning:** "Hipsters" are a cultural subculture stereotyped by their desire to be trendy, unique, and ahead of the curve—often liking bands, movies, or trends *before* they become mainstream or "cool."

**The punchline:** The hipster burned his mouth because he literally ate the pizza **before it was cool** (temperature-wise), which ironically aligns with his personality trait of liking everything before it is popular/cool.
Thread ID: 1
Step: 8
Next Node: ('generate_explanation',)

Topic: pizza

Joke:
Why did the hipster burn the

In [19]:
## Generating another configuration
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({"topic": "pasta"}, config=config2)

{'topic': 'pasta',
 'joke': [{'type': 'text',
   'text': "What is a carb-loader's favorite type of music?\n\n**A-spaghetti!**",
   'extras': {'signature': 'EpYBCpMBAVjVB3vaIAzstlDAktF3AbVNb+TFQ1XWl8NscUIGZYktPgAc/LXXn9ZHbChl3zxSxj74+t64d3aya07RBYs254xsZJppwUCLL7K04AOUKDFnCKHBHfU6TBvQrYTzhusv9GrlmbWN+L/z6FgNOTOB4Lg8cF8hZN3lNY8fK+aIltgzvtW4favtSusDbJYt2hJUCK56'}}],
 'explanation': [{'type': 'text',
   'text': 'This joke is a pun that relies on wordplay involving a musical genre and a popular type of pasta:\n\n1. **"Carb-loader"**: Someone who eats a lot of carbohydrates (like pasta, bread, or rice) in preparation for endurance sports like running a marathon.\n2. **"A-spaghetti"**: This is a play on the musical genre **"A cappella"** (singing without instrumental accompaniment). \n\nBy swapping out "cappella" for "spaghetti"—a quintessential carbohydrate—the joke humorously connects a runner\'s love for pasta-loading with a well-known musical term.',
   'extras': {'signature': 'EpYBCpMBAV

In [20]:
workflow.get_state(config=config2)

StateSnapshot(values={'topic': 'pasta', 'joke': [{'type': 'text', 'text': "What is a carb-loader's favorite type of music?\n\n**A-spaghetti!**", 'extras': {'signature': 'EpYBCpMBAVjVB3vaIAzstlDAktF3AbVNb+TFQ1XWl8NscUIGZYktPgAc/LXXn9ZHbChl3zxSxj74+t64d3aya07RBYs254xsZJppwUCLL7K04AOUKDFnCKHBHfU6TBvQrYTzhusv9GrlmbWN+L/z6FgNOTOB4Lg8cF8hZN3lNY8fK+aIltgzvtW4favtSusDbJYt2hJUCK56'}}], 'explanation': [{'type': 'text', 'text': 'This joke is a pun that relies on wordplay involving a musical genre and a popular type of pasta:\n\n1. **"Carb-loader"**: Someone who eats a lot of carbohydrates (like pasta, bread, or rice) in preparation for endurance sports like running a marathon.\n2. **"A-spaghetti"**: This is a play on the musical genre **"A cappella"** (singing without instrumental accompaniment). \n\nBy swapping out "cappella" for "spaghetti"—a quintessential carbohydrate—the joke humorously connects a runner\'s love for pasta-loading with a well-known musical term.', 'extras': {'signature': 'EpY

In [22]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': [{'type': 'text', 'text': "What is a carb-loader's favorite type of music?\n\n**A-spaghetti!**", 'extras': {'signature': 'EpYBCpMBAVjVB3vaIAzstlDAktF3AbVNb+TFQ1XWl8NscUIGZYktPgAc/LXXn9ZHbChl3zxSxj74+t64d3aya07RBYs254xsZJppwUCLL7K04AOUKDFnCKHBHfU6TBvQrYTzhusv9GrlmbWN+L/z6FgNOTOB4Lg8cF8hZN3lNY8fK+aIltgzvtW4favtSusDbJYt2hJUCK56'}}], 'explanation': [{'type': 'text', 'text': 'This joke is a pun that relies on wordplay involving a musical genre and a popular type of pasta:\n\n1. **"Carb-loader"**: Someone who eats a lot of carbohydrates (like pasta, bread, or rice) in preparation for endurance sports like running a marathon.\n2. **"A-spaghetti"**: This is a play on the musical genre **"A cappella"** (singing without instrumental accompaniment). \n\nBy swapping out "cappella" for "spaghetti"—a quintessential carbohydrate—the joke humorously connects a runner\'s love for pasta-loading with a well-known musical term.', 'extras': {'signature': 'Ep

In [23]:
for snapshot in workflow.get_state_history(config2):

    state = snapshot.values

    print("=" * 50)
    print("Thread ID:", snapshot.config["configurable"]["thread_id"])
    print("Step:", snapshot.metadata["step"])
    print("Next Node:", snapshot.next)

    print("\nTopic:", state.get("topic", "N/A"))

    joke = state.get("joke")
    if joke:
        print("\nJoke:")
        print(joke[0]["text"])

    explanation = state.get("explanation")
    if explanation:
        print("\nExplanation:")
        print(explanation[0]["text"])

    print("=" * 50)

Thread ID: 2
Step: 2
Next Node: ()

Topic: pasta

Joke:
What is a carb-loader's favorite type of music?

**A-spaghetti!**

Explanation:
This joke is a pun that relies on wordplay involving a musical genre and a popular type of pasta:

1. **"Carb-loader"**: Someone who eats a lot of carbohydrates (like pasta, bread, or rice) in preparation for endurance sports like running a marathon.
2. **"A-spaghetti"**: This is a play on the musical genre **"A cappella"** (singing without instrumental accompaniment). 

By swapping out "cappella" for "spaghetti"—a quintessential carbohydrate—the joke humorously connects a runner's love for pasta-loading with a well-known musical term.
Thread ID: 2
Step: 1
Next Node: ('generate_explanation',)

Topic: pasta

Joke:
What is a carb-loader's favorite type of music?

**A-spaghetti!**
Thread ID: 2
Step: 0
Next Node: ('generate_joke',)

Topic: pasta
Thread ID: 2
Step: -1
Next Node: ('__start__',)

Topic: N/A
